In [12]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

In [3]:
data = {
    "study_hours": [2, 3, 4, 5, 6, 7, 1, 3, 5, 8,
                    2, 4, 6, 7, 9, 1, 3, 5, 6, 8],

    "sleep_hours": [6, 7, 7, 8, 8, 7, 5, 6, 7, 8,
                    6, 7, 8, 7, 8, 5, 6, 7, 8, 8],

    "attendance": [65, 70, 75, 80, 85, 90, 60, 72, 82, 95,
                   68, 76, 88, 91, 96, 55, 70, 84, 89, 94],

    "exam_score": [50, 58, 65, 72, 80, 88, 45, 60, 74, 95,
                   53, 67, 83, 89, 98, 40, 57, 76, 85, 93]
}

df = pd.DataFrame(data)

print(df)

    study_hours  sleep_hours  attendance  exam_score
0             2            6          65          50
1             3            7          70          58
2             4            7          75          65
3             5            8          80          72
4             6            8          85          80
5             7            7          90          88
6             1            5          60          45
7             3            6          72          60
8             5            7          82          74
9             8            8          95          95
10            2            6          68          53
11            4            7          76          67
12            6            8          88          83
13            7            7          91          89
14            9            8          96          98
15            1            5          55          40
16            3            6          70          57
17            5            7          84      

In [20]:
X = df.drop(['exam_score'], axis=1)
y = df['exam_score']

In [21]:
X = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.float32)

torch.Size([20, 3]) torch.Size([20])


In [9]:
loader = DataLoader(dataset, batch_size=4, shuffle=True)

In [22]:
dataset = TensorDataset(X, y)

In [27]:
model = nn.Sequential(
    nn.Linear(3, 64),
        nn.ReLU(),

    nn.Dropout(p=0.4),
    nn.Linear(64, 128),
        nn.ReLU(),

    nn.Dropout(p=0.4),
    nn.Linear(128, 64),
        nn.ReLU(),

    nn.Dropout(p=0.4),
    nn.Linear(64, 1)
)

In [28]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

In [29]:
loss = nn.MSELoss()

In [38]:
num_epochs=50
best_val_loss = float('inf')
for epoch in range(num_epochs):
  model.train()
  total_loss = 0
  for X_batch, y_batch in loader:


    predictions = model(X_batch)
    calc_loss = loss(predictions, y_batch.unsqueeze(1))
    total_loss += calc_loss.item()
    optimizer.zero_grad()
    calc_loss.backward()
    optimizer.step()
  avg_loss = total_loss/len(loader)
  if epoch <20:
    if avg_loss < best_val_loss:
      best_val_loss = avg_loss
      checkpoint = {
          "epoch":epoch,
          "model_state_dict":model.state_dict(),
          "optim_state_dict":optimizer.state_dict()
      }
      torch.save(checkpoint, "checkpoint.pth")
    else:
      break


In [43]:
newer_model = nn.Sequential(
    nn.Linear(3, 64),
        nn.ReLU(),

    nn.Dropout(p=0.4),
    nn.Linear(64, 128),
        nn.ReLU(),

    nn.Dropout(p=0.4),
    nn.Linear(128, 64),
        nn.ReLU(),

    nn.Dropout(p=0.4),
    nn.Linear(64, 1)
)

In [44]:
new_optimizer = torch.optim.AdamW(newer_model.parameters(), lr=0.01)

In [ ]:
print(checkpoint)

In [42]:
checkpoint = torch.load("checkpoint.pth")

In [46]:
newer_model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [48]:
new_optimizer.load_state_dict(checkpoint["optim_state_dict"])

In [50]:
start_epoch = checkpoint["epoch"]
end_epoch = 50

In [56]:
loss_fn = nn.MSELoss()
best_loss = float('inf')
best_epoch = -1

In [57]:
for epoch in range(start_epoch, end_epoch):
  newer_model.train()
  total_loss = 0
  for X_batch, y_batch in loader:
    predictions = newer_model(X_batch)
    calc_loss = loss_fn(predictions, y_batch.unsqueeze(1))
    total_loss += calc_loss.item()
    optimizer.zero_grad()
    calc_loss.backward()
    optimizer.step()
  avg_loss = total_loss/len(loader)
  print(avg_loss)
  if avg_loss < best_loss:
    best_loss = avg_loss
    best_epoch = epoch
    torch.save(newer_model.state_dict(), "best_model.pth")


338.514111328125
354.47323455810545
174.9719451904297
362.4599151611328
327.6243011474609
327.2912261962891
379.4832290649414
309.416748046875
568.1577575683593
283.4429191589355
282.385075378418
360.3947868347168
205.23321228027345
336.25756683349607
203.86074676513672
327.4556915283203
259.14672622680666
216.03575630187987
443.6078079223633
507.03450164794924
243.7880889892578
219.51325454711915
383.1206115722656
344.4223297119141
424.3766830444336
235.78302612304688
299.50828857421874
257.61829833984376
258.01166648864745
378.4577239990234
214.45590362548828
252.7357971191406
393.1318298339844
318.8054992675781
240.12520370483398
583.6865814208984
395.6659240722656
160.6225383758545
284.99364318847654
337.2164245605469
290.6160949707031
209.4589370727539
291.58895111083984
305.67531433105466
166.97649459838868
174.7516876220703
221.13882751464843
133.29682388305665
269.89181289672854
333.5099182128906


In [64]:
state_dict = torch.load("best_model.pth")
newer_model.load_state_dict(state_dict)

count = 0
for key, value in state_dict.items():
  if count > 1:
    break
  print(key, value)
  count += 1

<class 'collections.OrderedDict'>
0.weight tensor([[-2.6807e-01,  1.3098e-01, -3.3736e-01],
        [-3.4509e-01,  4.1051e-01, -3.0291e-01],
        [ 2.8012e-01,  7.3039e-02, -4.5857e-02],
        [ 4.6922e-01, -2.5987e-01,  8.2184e-02],
        [-2.6786e-01,  4.3364e-01, -3.5429e-01],
        [ 6.1236e-01, -2.4553e-01, -3.6493e-02],
        [ 2.5856e-01,  2.5274e-01, -1.6517e-01],
        [ 4.7347e-01,  7.8898e-02, -2.8787e-01],
        [ 3.2809e-01, -2.9741e-01, -5.1233e-02],
        [-4.1079e-01, -7.5435e-02, -3.3993e-01],
        [ 1.0429e-01,  1.9617e-01, -3.6129e-01],
        [-1.2841e-02, -6.5306e-01,  4.0702e-02],
        [ 1.0826e-01, -3.5830e-01, -3.5627e-01],
        [-3.6630e-01,  2.3350e-01, -3.3265e-01],
        [-1.6391e-01,  2.5740e-02, -2.1192e-01],
        [-2.0477e-01, -7.0265e-01,  5.9643e-02],
        [ 6.3208e-02, -4.5113e-01, -1.6723e-01],
        [ 5.5618e-01, -5.6027e-01,  9.1818e-02],
        [ 1.8655e-01,  2.4058e-01, -4.1717e-01],
        [-3.5533e-02, -1.6